#GCP Authentication & Configuration
Switch on notebook access in secrets as needed.

In [ ]:
import os
from google.colab import auth, userdata

# --- Authentication ---
auth.authenticate_user()

# --- GCP Configuration ---
PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]

os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID


assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

# Helper functions

In [ ]:
import pandas as pd
from google.cloud import bigquery, storage

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
DATED = f"{PROJECT_ID}.tennis_clean.matches_dated"
AB = f"{PROJECT_ID}.tennis_clean.matches_ab"

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.width", 200)

# Build table

In [ ]:
SALT = "ab-v1"     # part of the assignment rule; changing it redraws A/B

CONTEXT = ["match_id", "tourney_id", "tourney_name", "name_key", "tourney_date", "season",
           "tourney_level", "surface", "draw_size", "best_of", "round", "event_format",
           "match_date_min", "match_date_max"]
PLAYER_FIELDS = ["id", "name", "seed", "entry", "hand", "ht", "ioc", "age", "rank", "rank_points"]
STAT_FIELDS = ["ace", "df", "svpt", "1stIn", "1stWon", "2ndWon", "SvGms", "bpSaved", "bpFaced"]

parts = []
for f in PLAYER_FIELDS:                                   # known before the match
    parts.append(f"IF(a_is_winner, winner_{f}, loser_{f}) AS a_{f}")
    parts.append(f"IF(a_is_winner, loser_{f}, winner_{f}) AS b_{f}")
for f in STAT_FIELDS:                                     # only known after the match
    parts.append(f"IF(a_is_winner, w_{f}, l_{f}) AS post_a_{f}")
    parts.append(f"IF(a_is_winner, l_{f}, w_{f}) AS post_b_{f}")

sql = f"""
CREATE OR REPLACE TABLE `{AB}`
PARTITION BY DATE_TRUNC(tourney_date, MONTH)
CLUSTER BY tourney_id
OPTIONS (description = 'One row per match: player A vs player B (A chosen by hash of match_id), label a_won. Columns starting post_ are only known after the match.')
AS
WITH base AS (
  SELECT *, MOD(FARM_FINGERPRINT(CONCAT('{SALT}|', match_id)), 2) = 0 AS a_is_winner
  FROM `{DATED}`
)
SELECT {", ".join(CONTEXT)},
       {", ".join(parts)},
       IF(a_is_winner, 1, 0) AS a_won,
       score AS post_score,
       minutes AS post_minutes,
       is_walkover AS post_is_walkover,
       is_retirement AS post_is_retirement,
       is_default AS post_is_default
FROM base
"""
job = bq.query(sql); job.result()
print("Built", AB, "| bytes processed:", f"{job.total_bytes_processed:,}")

# Checks

In [ ]:
cols = [f.name for f in bq.get_table(AB).schema]
print("columns:", len(cols))
print("winner/loser columns left:", [c for c in cols if "winner" in c or "loser" in c])     # expect []
print("post_ columns:", [c for c in cols if c.startswith("post_")])

print(run(f"""SELECT (SELECT COUNT(*) FROM `{AB}`) AS ab_rows,
                     (SELECT COUNT(*) FROM `{DATED}`) AS dated_rows,
                     (SELECT COUNT(DISTINCT match_id) FROM `{AB}`) AS unique_match_ids"""))

# Every A/B value must trace back to the winner or loser value it came from
checks = [f"(a.a_{f} IS NOT DISTINCT FROM IF(a.a_won = 1, m.winner_{f}, m.loser_{f}) "
          f"AND a.b_{f} IS NOT DISTINCT FROM IF(a.a_won = 1, m.loser_{f}, m.winner_{f}))" for f in PLAYER_FIELDS]
print(run(f"""
SELECT COUNTIF(NOT ({" AND ".join(checks)})) AS inconsistent_rows,
       COUNTIF(a.a_id = a.b_id) AS same_player_both_sides
FROM `{AB}` a JOIN `{DATED}` m USING (match_id)
"""))

# Checking the assignment is balanced and independent of the result

In [ ]:
print(run(f"SELECT COUNT(*) AS n, ROUND(AVG(a_won), 4) AS share_a_won FROM `{AB}`"))
print(run(f"SELECT season, COUNT(*) AS n, ROUND(AVG(a_won), 3) AS share_a_won FROM `{AB}` GROUP BY 1 ORDER BY 1"))
print(run(f"SELECT tourney_level, surface, COUNT(*) AS n, ROUND(AVG(a_won), 3) AS share_a_won FROM `{AB}` GROUP BY 1, 2 ORDER BY 1, 2"))

# Does being 'A' depend on rank? And does the signal point the right way?
print(run(f"""
SELECT COUNTIF(a_rank < b_rank) AS a_better_ranked,
       COUNTIF(a_rank > b_rank) AS b_better_ranked,
       ROUND(AVG(IF(a_rank < b_rank, a_won, NULL)), 3) AS better_ranked_wins_when_A,
       ROUND(AVG(IF(a_rank > b_rank, 1 - a_won, NULL)), 3) AS better_ranked_wins_when_B
FROM `{AB}` WHERE a_rank IS NOT NULL AND b_rank IS NOT NULL
"""))

# Save parquet copy and manifest
A snapshot of the reshaped table before features exist.

In [ ]:
import json, hashlib
from datetime import datetime, timezone

bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
EXPORT_PREFIX = "processed/matches_ab/v1/"

if any(True for _ in bucket.list_blobs(prefix=EXPORT_PREFIX, max_results=1)):
    print("v1 export already exists; not overwriting. Save any change as v2.")
else:
    bq.extract_table(AB, f"gs://{BUCKET_NAME}/{EXPORT_PREFIX}part-*.parquet",
                     job_config=bigquery.ExtractJobConfig(destination_format="PARQUET")).result()

files = [{"file": b.name, "bytes": b.size, "sha256": hashlib.sha256(b.download_as_bytes()).hexdigest()}
         for b in bucket.list_blobs(prefix=EXPORT_PREFIX)]
rows = run(f"SELECT COUNT(*) AS n FROM `{AB}`")["n"][0]
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
bucket.blob(f"manifests/matches_ab_v1_{stamp}.json").upload_from_string(
    json.dumps({"table": "tennis_clean.matches_ab", "rows": int(rows), "salt": SALT, "files": files}, indent=1),
    content_type="application/json")
print(files)